In [32]:
import json
import pandas as pd
from pathlib import Path

In [ ]:
base = Path().cwd()

data_path = base / "data"
results_path = base / "results"
chat_file = "sensor-actuator_single_gpt-4o-mini_n-2_acc-0.511_04.12.2025-10:37:05.json"
chat_file = results_path / chat_file

chat = json.loads(chat_file.read_text())

responses = chat["responses"]

In [47]:
sensor_df = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df = sensor_df[["requirement"]].copy()

actuator_df = pd.read_excel(data_path / "actuator_requirements.xlsx")

In [65]:
actuator_df.merge(sensor_df, how="inner", left_on="Requirement", right_on="requirement")[["Requirement"]].shape

(0, 1)

In [48]:
responses_df = pd.DataFrame(responses)

In [53]:
sensor_responses = sensor_df.merge(responses_df, how="left", left_on="requirement", right_on="requirement")[["requirement", "target_actuator","ai_answer"]]
sensor_responses.isna().sum()

requirement        0
target_actuator    0
ai_answer          0
dtype: int64

In [54]:
actuator_responses = actuator_df.merge(responses_df, how="left", left_on="Requirement", right_on="requirement")[["Requirement", "target_actuator","ai_answer"]]
actuator_responses.isna().sum()

Requirement        0
target_actuator    0
ai_answer          0
dtype: int64

In [60]:
actuator_responses.shape[0] + sensor_responses.shape[0]

182

In [61]:
responses_df.shape

(174, 10)

In [70]:
sensor_incorrect = sensor_responses.loc[sensor_responses["ai_answer"] != sensor_responses["target_actuator"]]
sensor_incorrect.sample(5)

,requirement,target_actuator,ai_answer
93,The steering torque system must include an aut...,0,1
126,The vehicle control systems must have the abil...,0,1
40,"In case of steering system malfunction, an eme...",0,1
29,"In the absence of power assist, the manual ste...",0,1
92,The system must include a failure mode analysi...,0,1


In [72]:
actuator_incorrect = actuator_responses.loc[actuator_responses["ai_answer"] != actuator_responses["target_actuator"]]
actuator_incorrect

,Requirement,target_actuator,ai_answer
8,The electronic throttle control shall incorpor...,1,0
10,The vehicle system shall verify successful eng...,1,0
12,The engine control must transition into a limp...,1,0
33,All signal processes must be isolated with a n...,1,0
38,The system shall be designed so that single er...,1,0


In [73]:
sensor_df_full = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df_incorrect = sensor_df_full.merge(sensor_incorrect, how="inner", on="requirement")

In [78]:
sensor_df_incorrect["num_sensors"] = sensor_df_incorrect.drop(columns=["requirement", "target_actuator", "ai_answer"]).sum(axis=1)

In [80]:
sensor_df_incorrect["num_sensors"].value_counts()

num_sensors
1    64
2    16
Name: count, dtype: int64

In [81]:
sensor_df_incorrect.to_excel(data_path / "sensor_incorrect.xlsx", index=False)
actuator_incorrect.to_excel(data_path / "actuator_incorrect.xlsx", index=False)